# 03 — Referências e comparação de classificadores
Usamos os conjuntos por utilizador do notebook 02. A comparação utiliza cinco folds de StratifiedGroupKFold **apenas no treino**, com os mesmos grupos e semente para todos os modelos. A Pipeline completa é ajustada dentro de cada fold. A validação separada permite analisar a transferência para outros utilizadores.
O F1 de Suspicious é o critério principal; Average Precision é secundário. Precision e Recall são interpretados com as contagens de erros. Quando não existem previsões positivas, a Precision é indefinida: o código devolve zero por convenção e assinala essa situação.
Os resultados iniciais usam a regra predict de cada algoritmo. A otimização e um eventual ajuste de limiar serão feitos exclusivamente no desenvolvimento. O teste final não é avaliado neste notebook.

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/cybercrime_forensic_dataset.csv').exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.experiencias import executar_experiencia, tabela_resultados, guardar_resultados
resultados = []

## Referências simples
A referência maioritária obtém accuracy elevada sem detetar qualquer registo suspeito. Prever sempre Suspicious atinge recall total à custa de sinalizar todos os registos normais. As pontuações constantes produzem ROC-AUC de 0,5 e Average Precision igual à prevalência positiva.
Estas regras são implementadas com DummyClassifier, usando as mesmas entradas e protocolo dos candidatos, para tornar explícito o que significa melhorar.

In [2]:
for nome in ["maioritaria", "sempre_suspeita"]:
    resultado = executar_experiencia(nome)
    resultados.append(resultado)
display(tabela_resultados(resultados).round(4))

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
maioritaria,0.000,0.0000,0.000,0.0000,0.0000,0.0,0.8369,0.5,0.1631,0,241
sempre_suspeita,0.285,0.0007,0.285,0.2804,0.1631,1.0,0.1631,0.5,0.1631,1237,0


## Regressão logística
Combina linearmente as características e aplica uma função logística para estimar a classe positiva. É uma referência explicável para relações aproximadamente lineares. Usamos C=1, máximo de 2 000 iterações e pesos balanced calculados apenas no treino de cada fold, para estudar o desequilíbrio. A escala é aprendida dentro da Pipeline. A regularização limita coeficientes extremos; interações complexas podem não ser representadas.

In [3]:
resultado = executar_experiencia("logistica")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
logistica,0.2368,0.0104,0.2946,0.2448,0.1636,0.4855,0.5115,0.5142,0.1671,598,124


Diferença média de F1 CV face à melhor referência: -0.0482
F1 treino / validação: 0.2946 0.2448
Erros de validação: FP = 598 ; FN = 124
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Árvore de decisão
Aplica regras sucessivas de separação e permite relações não lineares e interações. Esta experiência usa CART, critério Gini, profundidade máxima 5, pelo menos 20 amostras por folha e pesos balanced. O limite de complexidade reduz a tendência para memorizar dados. ID3, C4.5 e CART são famílias de árvores; a implementação usada é CART para classificação, não regressão. Uma única árvore pode ter elevada variância.

In [4]:
resultado = executar_experiencia("arvore")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
arvore,0.2478,0.0424,0.302,0.2819,0.1696,0.834,0.3072,0.5321,0.1743,984,40


Diferença média de F1 CV face à melhor referência: -0.0373
F1 treino / validação: 0.302 0.2819
Erros de validação: FP = 984 ; FN = 40
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Random Forest — bagging
Combina 100 árvores treinadas com amostras bootstrap e subconjuntos de características, reduzindo a variância de uma árvore isolada. Usamos profundidade máxima 8, mínimo de 5 registos por folha, pesos balanced e semente 42. Esta abordagem representa ensemble learning por bagging. Permite interações, mas não cria sinal preditivo que não exista; a diferença entre treino e validação ajuda a discutir sobreajustamento.

In [5]:
resultado = executar_experiencia("random_forest")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
random_forest,0.2283,0.0304,0.4582,0.2169,0.1522,0.3776,0.5555,0.495,0.156,507,150


Diferença média de F1 CV face à melhor referência: -0.0568
F1 treino / validação: 0.4582 0.2169
Erros de validação: FP = 507 ; FN = 150
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## k-Nearest Neighbors
Classifica por votação dos 15 vizinhos mais próximos, com pesos uniformes. Permite fronteiras locais sem assumir uma relação linear. A normalização evita que a escala de um campo domine as distâncias. A representação one-hot e a dimensionalidade podem tornar as distâncias menos informativas; a classe maioritária pode dominar a votação. Não é alterada a distribuição do teste.

In [6]:
resultado = executar_experiencia("knn")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
knn,0.0027,0.006,0.0054,0.0,0.0,0.0,0.8363,0.4984,0.1685,1,241


Diferença média de F1 CV face à melhor referência: -0.2824
F1 treino / validação: 0.0054 0.0
Erros de validação: FP = 1 ; FN = 241
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Naïve Bayes gaussiano
Usa o teorema de Bayes e uma hipótese simplificadora de independência condicional das entradas. GaussianNB admite variáveis contínuas, incluindo valores negativos após escala; MultinomialNB não é adequado a esta representação. As categorias one-hot e os pares temporais não satisfazem necessariamente a hipótese de independência. Usamos priors iguais para observar uma opção sensível ao desequilíbrio, fixada antes da avaliação, e o var_smoothing predefinido de 1e-9.

In [7]:
resultado = executar_experiencia("naive_bayes")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
naive_bayes,0.2463,0.01,0.2922,0.2522,0.1646,0.5394,0.4783,0.506,0.1766,660,111


Diferença média de F1 CV face à melhor referência: -0.0388
F1 treino / validação: 0.2922 0.2522
Erros de validação: FP = 660 ; FN = 111
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Support Vector Machine
Procura uma fronteira com margem ampla; o kernel RBF permite separação não linear. Usamos C=1, gamma=scale e pesos balanced. As entradas são escaladas dentro de cada fold. O modelo fornece decision_function para ROC-AUC e Average Precision: estas pontuações não são probabilidades calibradas. O custo e a sensibilidade aos parâmetros são limitações relevantes.

In [8]:
resultado = executar_experiencia("svm")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
svm,0.2402,0.035,0.5152,0.2101,0.1496,0.3527,0.5677,0.4868,0.1574,483,156


Diferença média de F1 CV face à melhor referência: -0.0449
F1 treino / validação: 0.5152 0.2101
Erros de validação: FP = 483 ; FN = 156
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Gradient Boosting — boosting
Acrescenta árvores pequenas sequencialmente para reduzir a função de perda, ao contrário das árvores bootstrap do bagging. Usamos 100 etapas, learning_rate=0,05 e profundidade 2. Esta experiência mantém a distribuição e os pesos predefinidos para mostrar o comportamento sem ponderação de classe. Pode aprender interações, mas também ajustar ruído; exige regularização e validação.

In [9]:
resultado = executar_experiencia("gradient_boosting")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
gradient_boosting,0.0,0.0,0.0,0.0,0.0,0.0,0.8363,0.5056,0.1581,1,241


Diferença média de F1 CV face à melhor referência: -0.285
F1 treino / validação: 0.0 0.0
Erros de validação: FP = 1 ; FN = 241
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## XGBoost
Implementa boosting de árvores com regularização e construção eficiente. A inclusão cobre a abordagem assinalada com M no Roadmap e permite comparação com Gradient Boosting e Random Forest. Usamos 100 árvores, profundidade 3, learning_rate=0,05 e reg_lambda=1. O peso positivo 5 é uma opção fixa motivada pelo desequilíbrio já conhecido na EDA, não é calculado no teste nem ajustado pelos seus resultados. Uma implementação eficiente não garante capacidade de generalização.

In [10]:
resultado = executar_experiencia("xgboost")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferença média de F1 CV face à melhor referência:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / validação:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de validação: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
xgboost,0.2394,0.024,0.3604,0.2581,0.1693,0.5436,0.4905,0.4972,0.1553,643,110


Diferença média de F1 CV face à melhor referência: -0.0457
F1 treino / validação: 0.3604 0.2581
Erros de validação: FP = 643 ; FN = 110
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Registo de resultados de desenvolvimento
O ficheiro pequeno seguinte contém métricas reais, não dados processados nem modelos. As referências e candidatos são avaliados apenas no desenvolvimento. A variabilidade dos folds é descritiva e não constitui um intervalo de confiança de incidentes independentes.

In [11]:
display(tabela_resultados(resultados).round(4))
guardar_resultados(resultados)
print("Resultados guardados em resultados/desenvolvimento.json; teste final não avaliado.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
maioritaria,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8369,0.5000,0.1631,0,241
sempre_suspeita,0.2850,0.0007,0.2850,0.2804,0.1631,1.0000,0.1631,0.5000,0.1631,1237,0
logistica,0.2368,0.0104,0.2946,0.2448,0.1636,0.4855,0.5115,0.5142,0.1671,598,124
arvore,0.2478,0.0424,0.3020,0.2819,0.1696,0.8340,0.3072,0.5321,0.1743,984,40
random_forest,0.2283,0.0304,0.4582,0.2169,0.1522,0.3776,0.5555,0.4950,0.1560,507,150
knn,0.0027,0.0060,0.0054,0.0000,0.0000,0.0000,0.8363,0.4984,0.1685,1,241
naive_bayes,0.2463,0.0100,0.2922,0.2522,0.1646,0.5394,0.4783,0.5060,0.1766,660,111
svm,0.2402,0.0350,0.5152,0.2101,0.1496,0.3527,0.5677,0.4868,0.1574,483,156
gradient_boosting,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8363,0.5056,0.1581,1,241


Resultados guardados em resultados/desenvolvimento.json; teste final não avaliado.
